## **Détection du langage offensif en Arabizi**

**Partie 1 – Exploration et prétraitement**

In [ ]:
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    classification_report, confusion_matrix
)

from sklearn.naive_bayes import MultinomialNB, ComplementNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier

In [ ]:
df = pd.read_csv("Arabizi-Off_Lang_Dataset.csv")
df.head()
df = df.rename(columns={
    "Text": "text",
    "Generic Class": "label",
    "Dialect": "dialect"
})

df["label"] = df["label"].map({
    "offensive": "Offensive",
    "non-offensive": "Non-Offensive"
})

df["dialect"] = df["dialect"].map({
    "DZ": "Algerian",
    "LB": "Lebanese"
})

df = df.dropna(subset=["text", "label", "dialect"])
df["id"] = range(1, len(df) + 1)

df = df[["id", "text", "label", "dialect"]]
df.head()


In [ ]:
print("Taille du corpus :", len(df))
print(df["label"].value_counts(normalize=True))
print(df["dialect"].value_counts())
df.sample(5)

In [ ]:
#Prétraitement du texte
def preprocess(text):
    text = text.lower()
    text = re.sub(r"http\S+|www\S+", "", text)   # URLs
    text = re.sub(r"@\w+", "", text)             # handles
    text = re.sub(r"[^a-z0-9\s]", " ", text)     # ponctuation
    text = re.sub(r"\s+", " ", text).strip()
    return text

df["clean_text"] = df["text"].apply(preprocess)

**Partie 2 – Modèles classiques**

In [ ]:
X = df["clean_text"]
y = df["label"]
dialects = df["dialect"]

X_train, X_temp, y_train, y_temp, d_train, d_temp = train_test_split(
    X, y, dialects, test_size=0.3, stratify=y, random_state=42
)

X_dev, X_test, y_dev, y_test, d_dev, d_test = train_test_split(
    X_temp, y_temp, d_temp, test_size=0.5, stratify=y_temp, random_state=42
)
# TF-IDF (mots + caractères)
tfidf = TfidfVectorizer(
    analyzer="char",
    ngram_range=(3,5),
    min_df=3,
    max_features=5000
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_dev_tfidf = tfidf.transform(X_dev)
X_test_tfidf = tfidf.transform(X_test)

#Modèles classiques
models = {
    "Multinomial NB": MultinomialNB(),
    "Complement NB": ComplementNB(),
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Linear SVM": LinearSVC(),
    "Random Forest": RandomForestClassifier(n_estimators=200)
}

results = []

for name, model in models.items():
    model.fit(X_train_tfidf, y_train)
    y_pred = model.predict(X_test_tfidf)

    acc = accuracy_score(y_test, y_pred)
    p, r, f, _ = precision_recall_fscore_support(
        y_test, y_pred, average="macro"
    )

    results.append([name, acc, p, r, f])

    print("\n", name)
    print(classification_report(y_test, y_pred))
results_df = pd.DataFrame(
    results,
    columns=["Model", "Accuracy", "Precision", "Recall", "Macro F1"]
)
results_df

#Résultats par dialecte
def eval_by_dialect(model):
    res = {}
    for dialect in ["Algerian", "Lebanese"]:
        idx = np.where(d_test.values == dialect)[0]

        X_d = X_test_tfidf[idx]
        y_true = y_test.iloc[idx]
        y_pred = model.predict(X_d)

        _, _, f1, _ = precision_recall_fscore_support(
            y_true, y_pred, average="macro", zero_division=0
        )
        res[dialect] = f1
    return res
for name, model in models.items():
    print(name, eval_by_dialect(model))



svm = models["Linear SVM"]
y_pred = svm.predict(X_test_tfidf)

cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt="d",
            xticklabels=["Non-Off", "Off"],
            yticklabels=["Non-Off", "Off"])
plt.title("Confusion Matrix – SVM")
plt.show()

errors = df.loc[X_test.index]
errors = errors.assign(pred=y_pred)

false_negatives = errors[
    (errors["label"] == "Offensive") &
    (errors["pred"] == "Non-Offensive")
]

false_negatives.sample(5)


In [ ]:
import numpy as np
import pandas as pd

# y_test : vraies étiquettes ("Offensive"/"Non-Offensive")
# y_pred : prédictions SVM
# d_test : dialectes sur le test ("Algerian"/"Lebanese")

def fp_fn_by_dialect(y_true, y_pred, dialects, positive_label="Offensive"):
    results = {}
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    dialects = np.array(dialects)

    for dialect in sorted(np.unique(dialects)):
        mask = dialects == dialect
        yt = y_true[mask]
        yp = y_pred[mask]

        tp = np.sum((yt == positive_label) & (yp == positive_label))
        fp = np.sum((yt != positive_label) & (yp == positive_label))
        fn = np.sum((yt == positive_label) & (yp != positive_label))
        tn = np.sum((yt != positive_label) & (yp != positive_label))

        n = len(yt)
        results[dialect] = {
            "n_exemples": int(n),
            "TP": int(tp),
            "FP": int(fp),
            "FN": int(fn),
            "TN": int(tn),
            "taux_FP": fp / n if n > 0 else 0.0,
            "taux_FN": fn / n if n > 0 else 0.0,
        }
    return results

fp_fn_svm = fp_fn_by_dialect(y_test, y_pred, d_test, positive_label="Offensive")

# Affichage propre
pd.DataFrame(fp_fn_svm).T

**Partie 3 – Modèle BERT**

In [ ]:
pip install transformers datasets torch

In [ ]:
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding
)
from datasets import Dataset

# Tokenizer
tokenizer = AutoTokenizer.from_pretrained("alger-ia/dziribert")

# Label mapping
label_map = {"Non-Offensive": 0, "Offensive": 1}

# HuggingFace datasets
train_ds = Dataset.from_dict({
    "text": X_train.tolist(),
    "label": [label_map[l] for l in y_train]
})

test_ds = Dataset.from_dict({
    "text": X_test.tolist(),
    "label": [label_map[l] for l in y_test]
})

# Tokenization SANS padding
def tokenize(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=128
    )

train_ds = train_ds.map(tokenize, batched=True, remove_columns=["text"])
test_ds = test_ds.map(tokenize, batched=True, remove_columns=["text"])

#Data collator (padding dynamique)
data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer,
    padding="longest"
)

# Model
model = AutoModelForSequenceClassification.from_pretrained(
    "alger-ia/dziribert",
    num_labels=2
)

# Training arguments
args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    learning_rate=2e-5,
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    report_to="none"
)

# Trainer
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    tokenizer=tokenizer,
    data_collator=data_collator
)

# Train
trainer.train()

In [ ]:
# Récupérer l'historique de training du Trainer
logs = pd.DataFrame(trainer.state.log_history)

# Afficher les différentes colonnes disponibles
print(logs.columns)
display(logs.head())

# Courbe de loss (train vs validation)
plt.figure(figsize=(8, 5))

# Loss d'entraînement
train_loss = logs[logs["loss"].notna()][["step", "loss"]]
plt.plot(train_loss["step"], train_loss["loss"], label="Train loss")

# Loss de validation (eval_loss)
eval_loss = logs[logs["eval_loss"].notna()][["step", "eval_loss"]]
plt.plot(eval_loss["step"], eval_loss["eval_loss"], label="Validation loss")

plt.xlabel("Steps")
plt.ylabel("Loss")
plt.title("Courbes de loss (DziriBERT)")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# Prédictions BERT
preds = trainer.predict(test_ds)
y_pred_bert = np.argmax(preds.predictions, axis=1)

y_true = y_test.map({"Non-Offensive": 0, "Offensive": 1}).values

# Scores globaux
acc_bert = accuracy_score(y_true, y_pred_bert)
p_bert, r_bert, f1_bert, _ = precision_recall_fscore_support(
    y_true, y_pred_bert, average="macro"
)

print("DziriBERT - Global")
print("Accuracy :", acc_bert)
print("Precision:", p_bert)
print("Recall   :", r_bert)
print("Macro F1 :", f1_bert)

In [ ]:
cm_bert = confusion_matrix(y_true, y_pred_bert)

plt.figure(figsize=(4, 4))
sns.heatmap(
    cm_bert,
    annot=True,
    fmt="d",
    xticklabels=["Non-Offensive", "Offensive"],
    yticklabels=["Non-Offensive", "Offensive"],
    cmap="Blues"
)
plt.xlabel("Prédiction")
plt.ylabel("Vérité terrain")
plt.title("Matrice de confusion – DziriBERT")
plt.tight_layout()
plt.show()

In [ ]:
id2label = {0: "Non-Offensive", 1: "Offensive"}
y_pred_bert_labels = np.array([id2label[i] for i in y_pred_bert])

y_test_labels = np.array(y_test.values)


def fp_fn_by_dialect(y_true, y_pred, dialects, positive_label="Offensive"):
    results = {}
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    dialects = np.array(dialects)

    for dialect in sorted(np.unique(dialects)):
        mask = dialects == dialect
        yt = y_true[mask]
        yp = y_pred[mask]

        tp = np.sum((yt == positive_label) & (yp == positive_label))
        fp = np.sum((yt != positive_label) & (yp == positive_label))
        fn = np.sum((yt == positive_label) & (yp != positive_label))
        tn = np.sum((yt != positive_label) & (yp != positive_label))

        n = len(yt)
        results[dialect] = {
            "n_exemples": int(n),
            "TP": int(tp),
            "FP": int(fp),
            "FN": int(fn),
            "TN": int(tn),
            "taux_FP": fp / n if n > 0 else 0.0,
            "taux_FN": fn / n if n > 0 else 0.0,
        }
    return results

fp_fn_bert = fp_fn_by_dialect(y_test_labels, y_pred_bert_labels, d_test, positive_label="Offensive")

# Affichage
pd.DataFrame(fp_fn_bert).T

In [ ]:
id2label = {0: "Non-Offensive", 1: "Offensive"}
y_pred_bert_labels = np.array([id2label[i] for i in y_pred_bert])

y_test_labels = np.array(y_test.values)      # "Offensive"/"Non-Offensive"
d_test_array = np.array(d_test.values)       # "Algerian"/"Lebanese"

errors_bert = df.loc[X_test.index].copy()
errors_bert["pred_bert"] = y_pred_bert_labels

#Faux négatifs BERT
false_negatives_bert = errors_bert[
    (errors_bert["label"] == "Offensive") &
    (errors_bert["pred_bert"] == "Non-Offensive")
]

# Afficher des exemples
false_negatives_bert[["id", "text", "label", "dialect", "pred_bert"]].sample(
    5, random_state=42
)

In [ ]:
#COMPARAISON DES SCORES GLOBAUX
comparison = results_df.copy()
comparison.loc[len(comparison)] = [
    "DziriBERT",
    acc_bert,
    p_bert,
    r_bert,
    f1_bert
]

comparison


In [ ]:
def eval_bert_by_dialect(dialect_name):
    idx = np.where(d_test.values == dialect_name)[0]

    y_true_d = y_true[idx]
    y_pred_d = y_pred_bert[idx]

    _, _, f1, _ = precision_recall_fscore_support(
        y_true_d, y_pred_d, average="macro", zero_division=0
    )
    return f1

print("DziriBERT Algerian F1:", eval_bert_by_dialect("Algerian"))
print("DziriBERT Lebanese F1:", eval_bert_by_dialect("Lebanese"))

for name, model in models.items():
    print(name, eval_by_dialect(model))

print("DziriBERT:", {
    "Algerian": eval_bert_by_dialect("Algerian"),
    "Lebanese": eval_bert_by_dialect("Lebanese")
})

In [ ]:
df_test = df.loc[X_test.index].copy()

df_test["pred_svm"] = y_pred
df_test["pred_bert"] = y_pred_bert_labels


In [ ]:
#Cas où SVM réussit et BERT échoue
svm_ok_bert_ko = df_test[
    (df_test["label"] == df_test["pred_svm"]) &
    (df_test["label"] != df_test["pred_bert"])
]

# Voir quelques exemples
svm_ok_bert_ko[["text", "label", "pred_svm", "pred_bert", "dialect"]].sample(10, random_state=0)

In [ ]:
#Cas où BERT réussit et SVM échoue
bert_ok_svm_ko = df_test[
    (df_test["label"] == df_test["pred_bert"]) &
    (df_test["label"] != df_test["pred_svm"])
]

bert_ok_svm_ko[["text", "label", "pred_svm", "pred_bert", "dialect"]].sample(10, random_state=0)